# Deepfake Detection AI — Training in Google Colab

This notebook fine-tunes EfficientNet-B0 on labeled face images using the training code in this repository. **You must supply a dataset**; it is not included in GitHub.

## Dataset requirements
Put your labeled images in Google Drive with this exact structure:

```text
MyDrive/deepfake-data/
  train/fake/  train/real/
  val/fake/    val/real/
  test/fake/   test/real/
```

Use data you have permission to access. FaceForensics++ access is controlled by its terms and access request process: https://github.com/ondyari/FaceForensics. If converting videos to frames, split by source video before extracting frames to avoid train/test leakage. Keep the test split untouched until final evaluation.

In Colab, select **Runtime → Change runtime type → T4 GPU** if available. A CPU works for small experiments but will be slower.

In [ ]:
import os, torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Tip: enable a GPU in Runtime → Change runtime type for faster training.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/swathyjayakumar222-art/Deepfake-Detection-AI.git'
REPO_DIR = '/content/Deepfake-Detection-AI'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd /content/Deepfake-Detection-AI/backend
!pip install -r requirements.txt

In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/deepfake-data')
required = [DATA_DIR / split / label for split in ('train', 'val', 'test') for label in ('fake', 'real')]
missing = [str(path) for path in required if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        'Dataset folders are missing. Create/copy the structure from the first notebook cell. Missing: ' + ', '.join(missing)
    )
for split in ('train', 'val', 'test'):
    print(f'\\n{split.upper()}')
    for label in ('fake', 'real'):
        folder = DATA_DIR / split / label
        count = sum(1 for p in folder.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'})
        print(f'  {label}: {count} images')

## Train the model

The default is five epochs as a quick baseline. For a proper minor-project experiment, run longer after checking learning curves, compare a custom CNN against EfficientNet-B0, and document the exact dataset split and hardware. Don't report metrics from the training set as test performance.

In [ ]:
%cd /content/Deepfake-Detection-AI/backend
!python train.py --data-dir "$DATA_DIR" --epochs 5 --batch-size 16 --workers 2

In [ ]:
import json
metrics_path = Path('/content/Deepfake-Detection-AI/backend/artifacts/metrics.json')
if not metrics_path.exists():
    raise FileNotFoundError('Training did not produce metrics.json; inspect the training output above.')
report = json.loads(metrics_path.read_text())
print(json.dumps({
    'architecture': report['architecture'],
    'train_count': report['train_count'],
    'validation_count': report['validation_count'],
    'test_count': report['test_count'],
    'test_metrics': report['test_metrics']
}, indent=2))

In [ ]:
# Persist trained weights and metrics in your own Drive; do not commit large model weights to GitHub.
import shutil
OUTPUT_DIR = Path('/content/drive/MyDrive/deepfake-training-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/Deepfake-Detection-AI/backend/artifacts/best_model.pt', OUTPUT_DIR / 'best_model.pt')
shutil.copy2(metrics_path, OUTPUT_DIR / 'metrics.json')
print('Saved model and metrics to:', OUTPUT_DIR)

## What the metrics mean

- **Precision (fake):** among images predicted fake, how many were fake in the labeled test set.
- **Recall (fake):** among labeled fake images, how many the model detected.
- **F1 (fake):** balance between fake precision and recall.
- **ROC-AUC (fake):** ranking quality across probability thresholds.
- **Confusion matrix:** actual vs predicted class counts; rows and columns are ordered real, then fake.

A good score on one dataset does not prove generalization to other deepfake methods. This is an educational model, not forensic proof.